# Guardrail Agent with Tool Calling and Memory


In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_openai import ChatOpenAI
from nemoguardrails import LLMRails, RailsConfig
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

## 1. Load the guardrails configuration

In [ ]:
rails = LLMRails(RailsConfig.from_path("./config"))

## 2. Define the graph state

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    blocked: bool

graph_builder = StateGraph(State)

## 3. Set up the LLM and tools

In [ ]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.0
)

In [ ]:
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

tools = [multiply]
llm_with_tools = llm.bind_tools(tools)

## 4. Input guardrail

In [ ]:
async def input_guard(state: State):
    user_msg = state["messages"][-1].content

    result = await rails.generate_async(
        messages=[{"role": "user", "content": user_msg}]
    )

    # Update this check if your NeMo config uses a different refusal phrase.
    refused = "can't respond" in result["content"].lower()

    if refused:
        return {
            "messages": [("assistant", result["content"])],
            "blocked": True
        }

    return {"blocked": False}

## 5. Tool-calling LLM node

In [ ]:
def tool_calling_llm(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

## 6. Output guardrail

In [ ]:
async def output_guard(state: State):
    assistant_message = state["messages"][-1].content

    result = await rails.generate_async(
        messages=[{"role": "assistant", "content": assistant_message}]
    )

    refused = "can't respond" in result["content"].lower()

    if refused:
        return {
            "messages": [("assistant", result["content"])],
            "blocked": True
        }

    return {"blocked": False}

## 7. Connect the nodes and add MemorySaver



In [ ]:
def route_after_input(state: State):
    if state.get("blocked", False):
        return END
    return "tool_calling_llm"

def route_after_llm(state: State):
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "tools"
    return "output_guard"

graph_builder.add_node("input_guard", input_guard)
graph_builder.add_node("tool_calling_llm", tool_calling_llm)
graph_builder.add_node("tools", ToolNode(tools=tools))
graph_builder.add_node("output_guard", output_guard)

graph_builder.add_edge(START, "input_guard")
graph_builder.add_conditional_edges("input_guard", route_after_input)
graph_builder.add_conditional_edges("tool_calling_llm", route_after_llm)
graph_builder.add_edge("tools", "tool_calling_llm")
graph_builder.add_edge("output_guard", END)

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

## 8. Run a conversation

In [ ]:
config = {"configurable": {"thread_id": "conversation-1"}}

response = await graph.ainvoke(
    {"messages": [("user", "Hi! My name is Sam. Please remember that for this conversation.")],
     "blocked": False},
    config=config
)

for m in response["messages"]:
    m.pretty_print()

## 9. Continue the same conversation

In [ ]:
response = await graph.ainvoke(
    {"messages": [("user", "What name did I tell you?")],
     "blocked": False},
    config=config
)

for m in response["messages"]:
    m.pretty_print()